# Stage 2: Structural Parsing (Table-Aware, Multi-Quarter)

Processes **all** downloaded `full-submission.txt` files for each fund across
**multiple quarters** (up to 4 accessions per CIK).
Extracts the `<informationTable>` holdings XML, converts to pandas DataFrames,
prepends metadata headers, chunks tables to ≤ 300 words (fits within the
embedding model's 256-token window), and saves to `processed_holdings.json`
with `text` + `metadata` per chunk.

In [1]:
# ── Set Working Directory ─────────────────────────────────────────────────────
# Ensure working directory is the Model/ folder so all relative paths
# (./data/, ./vector_db/, etc.) resolve consistently across notebooks.
import os, pathlib

_cwd = pathlib.Path.cwd()
# If VS Code opened the workspace root, navigate into Model/
if not (_cwd / "data").exists() and (_cwd / "Model" / "data").exists():
    os.chdir(_cwd / "Model")

os.makedirs("data", exist_ok=True)
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\user\Documents\1fintech\GenAI\Individual Assignment 2\Model


In [2]:
!pip install beautifulsoup4 lxml tabulate pandas

You should consider upgrading via the 'C:\Users\user\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
import os
import re
import json
import pandas as pd
from bs4 import BeautifulSoup

# ── Configuration ────────────────────────────────────────────────────────────
FILINGS_BASE = "./data/13f_filings/sec-edgar-filings"
OUTPUT_FILE  = "./data/processed_holdings.json"
CHUNK_WORD_LIMIT = 300   # ≤256 tokens ≈ 300 words — fits embedding model window

# ── Dynamic CIK → fund-name registry ────────────────────────────────────────
VALID_CIKS_PATH = "./data/valid_ciks.json"

_FALLBACK_CIK_TO_FUND = {
    "0001067983": "Berkshire Hathaway Inc",
    "0001649339": "Scion Asset Management (Michael Burry)",
    "0001037389": "Renaissance Technologies",
    "0001006438": "Appaloosa Management",
    "0001350694": "Bridgewater Associates",
}

if os.path.exists(VALID_CIKS_PATH):
    with open(VALID_CIKS_PATH, "r", encoding="utf-8") as _f:
        CIK_TO_FUND = json.load(_f)
    print(f"Dynamic registry loaded: {len(CIK_TO_FUND)} fund(s) from {VALID_CIKS_PATH}")
else:
    CIK_TO_FUND = _FALLBACK_CIK_TO_FUND
    print("[WARN] valid_ciks.json not found — using hardcoded fallback (run Stage 1 first)")

COLUMNS = ["nameOfIssuer", "titleOfClass", "cusip", "value", "sshPrnamt", "investmentDiscretion"]

print("Config loaded.")

Dynamic registry loaded: 100 fund(s) from ./data/valid_ciks.json
Config loaded.


In [4]:
# ── Helper: extract metadata from raw SEC header text ────────────────────────
def extract_header_metadata(raw_text: str) -> dict:
    """
    Pulls COMPANY CONFORMED NAME, FILED AS OF DATE, and CONFORMED PERIOD OF REPORT
    from the SEC SGML header block at the top of full-submission.txt.
    """
    meta = {}

    m = re.search(r"COMPANY CONFORMED NAME:\s*(.+)", raw_text)
    meta["company_name"] = m.group(1).strip() if m else "UNKNOWN"

    m = re.search(r"FILED AS OF DATE:\s*(\d{8})", raw_text)
    if m:
        d = m.group(1)
        meta["filed_date"] = f"{d[:4]}-{d[4:6]}-{d[6:]}"
    else:
        meta["filed_date"] = "UNKNOWN"

    m = re.search(r"CONFORMED PERIOD OF REPORT:\s*(\d{8})", raw_text)
    if m:
        d = m.group(1)
        meta["period_of_report"] = f"{d[:4]}-{d[4:6]}-{d[6:]}"
    else:
        meta["period_of_report"] = "UNKNOWN"

    return meta

In [5]:
# ── Helper: parse informationTable XML → DataFrame ───────────────────────────────
def parse_information_table(raw_text: str) -> pd.DataFrame:
    """
    Locates the <informationTable> block inside the raw SEC submission text,
    parses it with BeautifulSoup (lxml-xml), and returns a DataFrame.

    FIX (Bridgewater): Some filers use a namespace prefix on every tag,
    e.g., <ns1:informationTable> and <ns1:infoTable>, and nest <sshPrnamt>
    inside <ns1:shrsOrPrnAmt>.  Namespace prefixes are stripped before
    parsing so the same logic works for all filers.
    """
    # Locate the informationTable block -- handles bare or prefixed tag names
    start_match = re.search(r'<[\w]*:?informationTable[\s>]', raw_text, re.IGNORECASE)
    if not start_match:
        return pd.DataFrame(columns=COLUMNS)
    start = start_match.start()

    end_match = re.search(r'</[\w]*:?informationTable>', raw_text[start:], re.IGNORECASE)
    if not end_match:
        return pd.DataFrame(columns=COLUMNS)
    end = start + end_match.end()

    xml_block = raw_text[start:end]

    # Strip namespace prefixes: <ns1:infoTable> -> <infoTable>, </ns1:field> -> </field>
    xml_block = re.sub(r'<(/?)[\w]+:([\w]+)', r'<\1\2', xml_block)
    # Remove xmlns declarations to prevent lxml-xml parser warnings
    xml_block = re.sub(r'\s+xmlns(?::\w+)?="[^"]*"', '', xml_block)

    soup = BeautifulSoup(xml_block, "lxml-xml")

    rows = []
    for entry in soup.find_all("infoTable"):
        def txt(tag, _e=entry):  # default arg binds entry at definition time
            node = _e.find(tag)
            return node.get_text(strip=True) if node else ""

        rows.append({
            "nameOfIssuer":         txt("nameOfIssuer"),
            "titleOfClass":         txt("titleOfClass"),
            "cusip":                txt("cusip"),
            "value":                txt("value"),
            # Works for both flat <sshPrnamt> and nested <shrsOrPrnAmt><sshPrnamt>
            "sshPrnamt":            txt("sshPrnamt"),
            "investmentDiscretion": txt("investmentDiscretion"),
        })

    return pd.DataFrame(rows, columns=COLUMNS)


In [6]:
# ── Helper: chunk a Markdown table if it exceeds CHUNK_WORD_LIMIT words ──────
OVERLAP_ROWS = 3  # rows shared between adjacent chunks — prevents boundary-split holdings

def chunk_markdown_table(header: str, df: pd.DataFrame, word_limit: int) -> list:
    """
    Converts the DataFrame to Markdown. If the result exceeds `word_limit` words,
    splits the DataFrame into smaller sub-tables and prepends:
      1. The metadata header to every chunk.
      2. The Markdown column-header row to every chunk.
    Adjacent chunks share OVERLAP_ROWS rows so holdings at chunk boundaries
    appear in at least one complete chunk context.
    Returns a list of (chunk_text, chunk_index, total_chunks) tuples.
    """
    # Approximate row word count by sampling a full markdown render
    full_md = df.to_markdown(index=False)
    full_text = f"{header}\n\n{full_md}"

    if len(full_text.split()) <= word_limit:
        return [(full_text, 1, 1)]

    # Estimate rows per chunk
    lines = full_md.splitlines()
    # Lines: [0]=col headers, [1]=separator, [2+]=data rows
    col_header_lines = lines[:2]            # keep for every chunk
    data_lines = lines[2:]                  # actual data rows

    words_per_data_line = max(
        sum(len(l.split()) for l in data_lines) / max(len(data_lines), 1), 1
    )
    header_words = len(header.split()) + sum(len(l.split()) for l in col_header_lines)
    rows_per_chunk = max(int((word_limit - header_words) / words_per_data_line), 1)

    # Build chunks with overlap
    chunks = []
    stride = max(rows_per_chunk - OVERLAP_ROWS, 1)
    chunk_starts = list(range(0, len(df), stride))
    num_chunks = len(chunk_starts)

    for i, start in enumerate(chunk_starts):
        end = min(start + rows_per_chunk, len(df))
        sub_df = df.iloc[start:end]
        sub_md = sub_df.to_markdown(index=False)
        chunk_text = f"{header}\n\n{sub_md}"
        chunks.append((chunk_text, i + 1, num_chunks))
        if end >= len(df):
            # Last chunk reached end of dataframe — stop
            break

    # Fix total_chunks count if we broke early
    actual_count = len(chunks)
    chunks = [(text, idx, actual_count) for text, idx, _ in chunks]

    return chunks

In [7]:
# ── Main processing loop (multi-quarter) ─────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

VALID_FILING_YEARS = {"2024", "2025", "2026"}
all_chunks    = []
stale_skipped = []
filing_count  = 0

for cik, fund_name in CIK_TO_FUND.items():
    fund_dir = os.path.join(FILINGS_BASE, cik, "13F-HR")
    if not os.path.exists(fund_dir):
        print(f"[SKIP] {fund_name} -- directory not found: {fund_dir}")
        continue

    accession_dirs = sorted(
        [d for d in os.listdir(fund_dir) if os.path.isdir(os.path.join(fund_dir, d))],
        reverse=True,
    )
    if not accession_dirs:
        print(f"[SKIP] {fund_name} -- no accession folder found")
        continue

    # ── Process ALL accession directories (multi-quarter) ────────────
    for accession_id in accession_dirs:
        submission_path = os.path.join(fund_dir, accession_id, "full-submission.txt")
        if not os.path.exists(submission_path):
            continue

        try:
            with open(submission_path, "r", encoding="utf-8", errors="replace") as f:
                raw_text = f.read()
        except OSError as exc:
            print(f"  [ERROR] Could not read {submission_path}: {exc}")
            continue

        # ── Extract metadata ─────────────────────────────────────────
        meta          = extract_header_metadata(raw_text)
        filing_date   = meta["filed_date"]
        period_report = meta["period_of_report"]
        reported_name = meta["company_name"]

        # ── Date guard ───────────────────────────────────────────────
        filing_year = filing_date[:4] if filing_date not in ("UNKNOWN", "") else "UNKNOWN"
        if filing_year not in VALID_FILING_YEARS:
            stale_skipped.append({"cik": cik, "fund_name": fund_name,
                                  "accession": accession_id, "filing_date": filing_date})
            continue

        filing_count += 1
        md_header = (
            f"## Fund: {fund_name}\n"
            f"**Reported Name:** {reported_name}  \n"
            f"**Filing Date:** {filing_date}  \n"
            f"**Period of Report:** {period_report}  \n"
            f"**Accession Number:** {accession_id}  \n"
            f"**CIK:** {cik}"
        )

        # ── Parse holdings table ─────────────────────────────────────
        df = parse_information_table(raw_text)
        if df.empty:
            print(f"  [WARN] No holdings rows: {fund_name} / {accession_id}")
            continue

        # ── Chunk and save ───────────────────────────────────────────
        chunks = chunk_markdown_table(md_header, df, CHUNK_WORD_LIMIT)

        for chunk_text, chunk_idx, total_chunks in chunks:
            all_chunks.append({
                "text": chunk_text,
                "metadata": {
                    "fund_name":        fund_name,
                    "cik":              cik,
                    "filing_date":      filing_date,
                    "period_of_report": period_report,
                    "accession_number": accession_id,
                    "chunk_index":      chunk_idx,
                    "total_chunks":     total_chunks,
                    "filing_type":      "13F-HR",
                },
            })

    print(f"  {fund_name}: {len(accession_dirs)} filing(s) processed")

if stale_skipped:
    print(f"\n{'='*60}")
    print(f"WARNING: {len(stale_skipped)} filing(s) skipped due to stale dates:")
    for s in stale_skipped[:20]:
        print(f"  - {s['fund_name']} / {s['accession']} filed {s['filing_date']}")
    if len(stale_skipped) > 20:
        print(f"  ... and {len(stale_skipped) - 20} more")

# ── Write output ─────────────────────────────────────────────────────────────
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"\n{'='*60}")
print(f"Done. {filing_count} filings → {len(all_chunks)} total chunks written to: {OUTPUT_FILE}")

  Berkshire Hathaway Inc (Warren Buffett): 4 filing(s) processed
  Scion Asset Management (Michael Burry): 4 filing(s) processed
  Bridgewater Associates (Ray Dalio): 4 filing(s) processed
  Vanguard Group Inc: 4 filing(s) processed
  Fidelity Management & Research (FMR): 4 filing(s) processed
  Geode Capital Management: 4 filing(s) processed
  Northern Trust Corp: 4 filing(s) processed
  Ariel Investments: 4 filing(s) processed
  Dimensional Fund Advisors: 4 filing(s) processed
  BNY Mellon Investment Management: 4 filing(s) processed
  WCM Investment Management: 4 filing(s) processed
  Morgan Stanley: 4 filing(s) processed
  Goldman Sachs Group: 4 filing(s) processed
  JPMorgan Chase & Co: 4 filing(s) processed
  Bank of America Corp: 4 filing(s) processed
  Wells Fargo & Company: 4 filing(s) processed
  UBS Group AG: 4 filing(s) processed
  Citigroup Inc: 4 filing(s) processed
  Barclays PLC: 4 filing(s) processed
  Ameriprise Financial Inc: 4 filing(s) processed
  Raymond James Fin

In [8]:
# ── Verification: print a summary of the output ───────────────────────────────
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total chunks in file: {len(data)}\n")
print(f"{'Fund':<40} {'Filings':>7} {'Chunks':>7}  Periods")
print("-" * 100)

from collections import defaultdict
summary = defaultdict(lambda: {"chunks": 0, "periods": set()})
for chunk in data:
    m = chunk["metadata"]
    key = m["fund_name"]
    summary[key]["chunks"] += 1
    summary[key]["periods"].add(m["period_of_report"])

for fund, info in sorted(summary.items()):
    periods = sorted(info["periods"])
    print(f"{fund:<40} {len(periods):>7} {info['chunks']:>7}  {', '.join(periods)}")

print(f"\n--- Chunk size stats ---")
word_counts = [len(c["text"].split()) for c in data]
print(f"Min: {min(word_counts)} words | Median: {sorted(word_counts)[len(word_counts)//2]} words | "
      f"Max: {max(word_counts)} words | Avg: {sum(word_counts)//len(word_counts)} words")

print(f"\n--- Sample chunk (first 500 chars) ---")
print(data[0]["text"][:500])

Total chunks in file: 95377

Fund                                     Filings  Chunks  Periods
----------------------------------------------------------------------------------------------------
AXA Financial Inc                              4       4  2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31
Advance Capital Management                     4      58  2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31
Alberta Investment Management Corp             4     182  2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31
Ameriprise Financial Inc                       4    5468  2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31
Ariel Investments                              4    1296  2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31
Arkansas Financial Group Inc                   4      30  2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31
Atria Wealth Solutions                         4    1269  2024-09-30, 2024-12-31, 2025-03-31, 2025-06-30
Auxano Advisors LLC                            4      54  2025-03-31,